Inconsistencies in data refer to discrepancies, contradictions, or irregularities in the dataset that affect its reliability and accuracy. These inconsistencies can arise due to various reasons such as data entry errors, integration of multiple sources, missing values, or incorrect formatting. Inconsistent data can lead to incorrect insights and poor decision-making.

**Types of Data Inconsistencies**
    
1. **Structural Inconsistencies:**

    - Mismatched column names or data types across datasets.

    - Irregular formatting (e.g., dates in different formats: DD/MM/YYYY vs. MM-DD-YYYY).

2. **Duplicate Data:**

    - Multiple records representing the same entity but with slight variations (e.g., same customer listed twice with different spellings).

3. **Contradictory Data:**

    - Conflicting information within the dataset (e.g., a person's age recorded as 25 in one record and 30 in another).

4. **Missing or Null Values:**

    - Fields left empty or containing NULL, affecting completeness.

5. **Inaccurate Data:**

    - Values that are incorrect or don't align with reality (e.g., negative age, invalid email addresses).

6. **Referential Integrity Issues:**

    - Foreign key relationships not maintained (e.g., an order references a customer ID that doesn’t exist in the customers table).

## Detect Inconsistencies

In [1]:
import pandas as pd
import numpy as np

# Create a sample dataset with inconsistencies
data = {
    'Customer ID': [101, 102, 103, 104, 105, 105],  # Duplicate ID
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Eve '],  # Extra space in Name
    'Age': [25, -30, 40, 29, None, 35],  # Negative and missing values
    'Email': ['alice@mail.com', 'bob@mail.com', 'charlie@mail.com', 'david@mail', 'eve@mail.com', 'eve@mail.com'],  # Invalid email
    'Join Date': ['2023-01-10', '01-12-2022', '2023-03-15', 'March 20, 2023', '2022-12-30', '2022-12-30'],  # Inconsistent date formats
    'Purchase Amount ($)': ['100', '200.5', 'N/A', 400, 150, '200'],  # Non-numeric values
}

df = pd.DataFrame(data)

df

,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($)
0,101,Alice,25.0,alice@mail.com,2023-01-10,100
1,102,Bob,-30.0,bob@mail.com,01-12-2022,200.5
2,103,Charlie,40.0,charlie@mail.com,2023-03-15,N/A
3,104,David,29.0,david@mail,"March 20, 2023",400
4,105,Eve,NaN,eve@mail.com,2022-12-30,150
5,105,Eve,35.0,eve@mail.com,2022-12-30,200


In [2]:
# Check for Duplicate Records

print("Duplicate Rows:")
df[df.duplicated()]

Duplicate Rows:


,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($)


In [3]:
# Check for Missing Values

print("Missing Values Count:")
df.isnull().sum()

Missing Values Count:


Customer ID            0
Name                   0
Age                    1
Email                  0
Join Date              0
Purchase Amount ($)    0
dtype: int64

In [4]:
# Check for Invalid Age Values (e.g., negative)

print("Invalid Age Values:")
df[df['Age'] < 0]

Invalid Age Values:


,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($)
1,102,Bob,-30.0,bob@mail.com,01-12-2022,200.5


In [5]:
# Check for Invalid Email Format

import re

def is_valid_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, str(email)))

df['Valid Email'] = df['Email'].apply(is_valid_email)
df[df['Valid Email'] == False]

,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($),Valid Email
3,104,David,29.0,david@mail,"March 20, 2023",400,False


In [6]:
# Check for Inconsistent Date Formats

df['Formatted Join Date'] = pd.to_datetime(df['Join Date'], errors='coerce')
print("Inconsistent Dates:")
df[df['Formatted Join Date'].isnull()]

Inconsistent Dates:


,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($),Valid Email,Formatted Join Date
1,102,Bob,-30.0,bob@mail.com,01-12-2022,200.5,True,NaT
3,104,David,29.0,david@mail,"March 20, 2023",400,False,NaT


In [7]:
# Convert Non-Numeric Values in 'Purchase Amount ($)'

df['Purchase Amount ($)'] = pd.to_numeric(df['Purchase Amount ($)'], errors='coerce')
print("Non-numeric values converted:")
df

Non-numeric values converted:


,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($),Valid Email,Formatted Join Date
0,101,Alice,25.0,alice@mail.com,2023-01-10,100.0,True,2023-01-10
1,102,Bob,-30.0,bob@mail.com,01-12-2022,200.5,True,NaT
2,103,Charlie,40.0,charlie@mail.com,2023-03-15,NaN,True,2023-03-15
3,104,David,29.0,david@mail,"March 20, 2023",400.0,False,NaT
4,105,Eve,NaN,eve@mail.com,2022-12-30,150.0,True,2022-12-30
5,105,Eve,35.0,eve@mail.com,2022-12-30,200.0,True,2022-12-30


## Fix Data Inconsistencies

In [8]:
# Remove Duplicate Records

df = df.drop_duplicates()

In [9]:
# handling missing values

df['Age'] = df['Age'].fillna(df['Age'].median())  # Fill missing age with median

In [10]:
# Fix Negative Age Values

df['Age'] = df['Age'].apply(lambda x: abs(x) if x < 0 else x)


In [11]:
# Remove Invalid Emails

df = df.loc[df['Valid Email'] == True]   # Keep only valid emails
df.drop(columns=['Valid Email'], inplace=True)  # Drop validation column

In [12]:
#  Standardize Date Format
df.loc[:, 'Join Date'] = pd.to_datetime(df['Join Date'], errors='coerce') # Convert all to datetime

In [13]:
# Convert 'Purchase Amount' to Numeric and Handle Errors

df.loc[:, 'Purchase Amount ($)'] = pd.to_numeric(df['Purchase Amount ($)'], errors='coerce')
df.loc[:, 'Purchase Amount ($)'] = df['Purchase Amount ($)'].fillna(df['Purchase Amount ($)'].median())  # Fill missing values

In [14]:
# Trim Extra Spaces in Name Column

df.loc[:, 'Name'] = df['Name'].str.strip()


In [15]:
# print("Cleaned Data:")
df


,Customer ID,Name,Age,Email,Join Date,Purchase Amount ($),Formatted Join Date
0,101,Alice,25.0,alice@mail.com,2023-01-10 00:00:00,100.0,2023-01-10
1,102,Bob,30.0,bob@mail.com,NaT,200.5,NaT
2,103,Charlie,40.0,charlie@mail.com,2023-03-15 00:00:00,175.0,2023-03-15
4,105,Eve,29.0,eve@mail.com,2022-12-30 00:00:00,150.0,2022-12-30
5,105,Eve,35.0,eve@mail.com,2022-12-30 00:00:00,200.0,2022-12-30
